# BEATs Fine-Tuning on AnnotatedVocalSet

Этот ноутбук обучает `BEATs` в трёх режимах:
- `full_tuning` - обучаются все параметры модели,
- `head_tuning` - обучается только классификационная голова,
- `lora_tuning` - обучаются LoRA-адаптеры + классификационная голова.

Целевая задача: **single-label классификация** (предсказание корректного класса `label`).


In [16]:
import json
import math
import os
import random
import sys
from contextlib import nullcontext
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import soundfile as sf
from sklearn.manifold import TSNE
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import GroupShuffleSplit
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm


In [17]:
def load_audio_robust(path: str) -> Tuple[torch.Tensor, int]:
    """Load audio without hard dependency on torchcodec.

    Returns:
        waveform: torch.Tensor [channels, time], float32
        sample_rate: int
    """
    # 1) Prefer soundfile to avoid torchaudio->torchcodec issues.
    try:
        wav_np, sr = sf.read(path, dtype="float32", always_2d=True)
        # soundfile: [time, channels] -> [channels, time]
        wav = torch.from_numpy(np.ascontiguousarray(wav_np.T))
        return wav, int(sr)
    except Exception as sf_err:
        # 2) Fallback to torchaudio loader.
        try:
            wav, sr = torchaudio.load(path)
            return wav.float(), int(sr)
        except Exception as ta_err:
            raise RuntimeError(
                f"Failed to load audio '{path}'. soundfile_error={sf_err}; torchaudio_error={ta_err}"
            ) from ta_err


In [18]:
def resolve_project_root() -> Path:
    p = Path.cwd().resolve()
    for c in [p, *p.parents]:
        if (c / "MainExperiments").exists() and (c / "datasets").exists():
            return c
    return p

PROJECT_ROOT = resolve_project_root()
BEATS_DIR = PROJECT_ROOT / "MainExperiments" / "BEATs"
if str(BEATS_DIR) not in sys.path:
    sys.path.insert(0, str(BEATS_DIR))

from BEATs import BEATs, BEATsConfig

print("PROJECT_ROOT:", PROJECT_ROOT)
print("BEATS_DIR:", BEATS_DIR)


PROJECT_ROOT: /home/evgeniy/Projects/VocalAssistant
BEATS_DIR: /home/evgeniy/Projects/VocalAssistant/MainExperiments/BEATs


In [19]:
# =========================
# Configuration
# =========================
PATCH_VERSION = "beats_notebook_patch_2026_04_21_v3"
print("PATCH_VERSION:", PATCH_VERSION)

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Выберите режим: full_tuning | head_tuning | lora_tuning
TRAINING_MODE = "head_tuning"
RUN_ALL_MODES = False
MODES = ["full_tuning", "head_tuning", "lora_tuning"]

# Можно явно задать CSV. Если None - будет взят первый найденный prepared_metadata.csv
METADATA_CSV = None
LABEL_COL = "label"
GROUP_COL = "group_id"  # чтобы снизить утечку между train/val
VAL_SIZE = 0.1

CHECKPOINT_PATH = BEATS_DIR / "BEATs_iter1.pt"
OUTPUT_BASE_DIR = PROJECT_ROOT / "MainExperiments" / "outputs" / "beats_annotated_vocalset"

TARGET_SR = 16000
MAX_AUDIO_SECONDS = 14.0  # ограничение длины для более стабильного обучения
BATCH_SIZE = 4
NUM_WORKERS = 4
USE_AMP = True
GRAD_CLIP_NORM = 1.0

EPOCHS_BY_MODE = {
    "full_tuning": 20,
    "head_tuning": 20,
    "lora_tuning": 20,
}

LR_BY_MODE = {
    "full_tuning": 1e-5,
    "head_tuning": 2e-4,
    "lora_tuning": 8e-5,
}
WEIGHT_DECAY = 1e-4

# LoRA config
LORA_R = 4
LORA_ALPHA = 32
LORA_DROPOUT = 0.2
LORA_TARGET_KEYWORDS = ("q_proj", "k_proj", "v_proj", "out_proj", "fc1", "fc2")

# Validation / logging
EVAL_EVERY_STEPS = 80  # валидация каждые N train-шагов

# Comet ML
COMET_ENABLED = True
COMET_PROJECT_NAME = "diploma"
COMET_WORKSPACE = None  # например: "your_workspace"
COMET_EXPERIMENT_PREFIX = "beats"
COMET_LOG_EVERY_N_STEPS = 5
COMET_AUTO_OFFLINE = False  # если нет COMET_API_KEY, логируем оффлайн


PATCH_VERSION: beats_notebook_patch_2026_04_21_v3


In [20]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
print("Device:", DEVICE)


Device: cuda


In [21]:
def resolve_metadata_csv(explicit_path: Optional[str] = None) -> Path:
    if explicit_path is not None:
        p = Path(explicit_path).expanduser().resolve()
        if not p.exists():
            raise FileNotFoundError(f"METADATA_CSV not found: {p}")
        return p

    candidates = [
        PROJECT_ROOT / "MainExperiments" / "outputs" / "wav2vec2_annotated_vocalset" / "wav2vec2_annotated_full_tuning" / "prepared_metadata.csv",
        PROJECT_ROOT / "MainExperiments" / "outputs" / "wavlm_annotated_vocalset" / "prepared_metadata.csv",
        PROJECT_ROOT / "MainExperiments" / "outputs" / "wav2vec2_annotated_vocalset_freezed_encoder" / "prepared_metadata.csv",
    ]
    for p in candidates:
        if p.exists():
            return p

    all_prepared = sorted((PROJECT_ROOT / "MainExperiments" / "outputs").glob("**/*annotated*/*prepared_metadata.csv"))
    if all_prepared:
        return all_prepared[0]

    raise FileNotFoundError(
        "Не найден prepared_metadata.csv для AnnotatedVocalSet. "
        "Укажите путь вручную в METADATA_CSV."
    )


def load_and_prepare_metadata(
    metadata_csv: Optional[str],
    label_col: str = "label",
    group_col: str = "group_id",
    val_size: float = 0.2,
    seed: int = 42,
) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, int], Dict[int, str], Path]:
    csv_path = resolve_metadata_csv(metadata_csv)
    meta = pd.read_csv(csv_path)

    required_cols = ["path", label_col, group_col]
    missing_cols = [c for c in required_cols if c not in meta.columns]
    if missing_cols:
        raise ValueError(f"В metadata отсутствуют колонки: {missing_cols}")

    meta = meta.dropna(subset=["path", label_col, group_col]).copy()
    meta["path"] = meta["path"].astype(str)

    # Оставляем только существующие аудиофайлы
    meta = meta[meta["path"].map(lambda p: Path(p).exists())].copy()
    meta = meta.reset_index(drop=True)

    labels = sorted(meta[label_col].unique().tolist())
    label2id = {lab: i for i, lab in enumerate(labels)}
    id2label = {i: lab for lab, i in label2id.items()}
    meta["label_id"] = meta[label_col].map(label2id)

    splitter = GroupShuffleSplit(n_splits=1, test_size=val_size, random_state=seed)
    train_idx, val_idx = next(splitter.split(meta, y=meta["label_id"], groups=meta[group_col]))

    train_df = meta.iloc[train_idx].reset_index(drop=True)
    val_df = meta.iloc[val_idx].reset_index(drop=True)

    print(f"metadata_csv: {csv_path}")
    print(f"rows total: {len(meta)} | train: {len(train_df)} | val: {len(val_df)}")
    print(f"num classes: {len(label2id)}")

    return train_df, val_df, label2id, id2label, csv_path


train_df, val_df, label2id, id2label, metadata_path = load_and_prepare_metadata(
    METADATA_CSV,
    label_col=LABEL_COL,
    group_col=GROUP_COL,
    val_size=VAL_SIZE,
    seed=SEED,
)

train_df.head(3)


metadata_csv: /home/evgeniy/Projects/VocalAssistant/MainExperiments/outputs/wav2vec2_annotated_vocalset/wav2vec2_annotated_full_tuning/prepared_metadata.csv
rows total: 2664 | train: 2386 | val: 278
num classes: 16


,stem,gender_ann,group_id,label,music_type_ann,vowel_ann,Total Duration,path,duration_sec,singer,gender,label_id
0,f2_arepggios_c_fast_forte_a,Female,f2,Fast_Articulated_Forte,Arpeggio,a,2.26395,/home/evgeniy/Projects/VocalAssistant/datasets...,2.274172,female2,female,2
1,f2_arepggios_c_fast_forte_e,Female,f2,Fast_Articulated_Forte,Arpeggio,e,2.29878,/home/evgeniy/Projects/VocalAssistant/datasets...,2.310249,female2,female,2
2,f2_arepggios_c_fast_forte_i,Female,f2,Fast_Articulated_Forte,Arpeggio,i,2.34522,/home/evgeniy/Projects/VocalAssistant/datasets...,2.346372,female2,female,2


In [22]:
class AnnotatedVocalSetDataset(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        target_sr: int = 16000,
        max_audio_seconds: Optional[float] = 8.0,
        random_crop: bool = False,
    ):
        self.df = df.reset_index(drop=True)
        self.target_sr = target_sr
        self.max_samples = int(max_audio_seconds * target_sr) if max_audio_seconds is not None else None
        self.random_crop = random_crop
        self._resamplers: Dict[int, torchaudio.transforms.Resample] = {}

    def __len__(self):
        return len(self.df)

    def _get_resampler(self, source_sr: int) -> torchaudio.transforms.Resample:
        if source_sr not in self._resamplers:
            self._resamplers[source_sr] = torchaudio.transforms.Resample(source_sr, self.target_sr)
        return self._resamplers[source_sr]

    def _load_waveform(self, path: str) -> torch.Tensor:
        wav, sr = load_audio_robust(path)
        if wav.size(0) > 1:
            wav = wav.mean(dim=0, keepdim=True)
        wav = wav.squeeze(0)

        if sr != self.target_sr:
            wav = self._get_resampler(sr)(wav.unsqueeze(0)).squeeze(0)

        # Нормализация пика
        peak = wav.abs().max().clamp(min=1e-6)
        wav = wav / peak

        if self.max_samples is not None and wav.numel() > self.max_samples:
            if self.random_crop:
                max_start = wav.numel() - self.max_samples
                start = int(torch.randint(0, max_start + 1, (1,)).item())
            else:
                start = (wav.numel() - self.max_samples) // 2
            wav = wav[start : start + self.max_samples]

        return wav.float()

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        waveform = self._load_waveform(row["path"])
        return {
            "waveform": waveform,
            "label": int(row["label_id"]),
            "path": row["path"],
        }


def collate_audio_batch(batch: List[Dict]) -> Dict[str, torch.Tensor]:
    lengths = [item["waveform"].numel() for item in batch]
    max_len = max(lengths)
    bsz = len(batch)

    waveforms = torch.zeros((bsz, max_len), dtype=torch.float32)
    padding_mask = torch.ones((bsz, max_len), dtype=torch.bool)  # True = pad
    labels = torch.zeros((bsz,), dtype=torch.long)

    for i, item in enumerate(batch):
        w = item["waveform"]
        n = w.numel()
        waveforms[i, :n] = w
        padding_mask[i, :n] = False
        labels[i] = item["label"]

    return {
        "waveforms": waveforms,
        "padding_mask": padding_mask,
        "labels": labels,
    }


train_ds = AnnotatedVocalSetDataset(
    train_df,
    target_sr=TARGET_SR,
    max_audio_seconds=MAX_AUDIO_SECONDS,
    random_crop=True,
)
val_ds = AnnotatedVocalSetDataset(
    val_df,
    target_sr=TARGET_SR,
    max_audio_seconds=MAX_AUDIO_SECONDS,
    random_crop=False,
)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    collate_fn=collate_audio_batch,
)
val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    collate_fn=collate_audio_batch,
)

print(f"train batches: {len(train_loader)} | val batches: {len(val_loader)}")


train batches: 597 | val batches: 70


In [23]:
class BEATsForSingleLabelClassification(nn.Module):
    """Обертка над BEATs для single-label классификации."""

    def __init__(self, checkpoint_path: Path, num_classes: int):
        super().__init__()
        checkpoint = torch.load(checkpoint_path, map_location="cpu")

        cfg = BEATsConfig(checkpoint["cfg"])
        cfg.finetuned_model = True
        cfg.predictor_class = num_classes

        self.beats = BEATs(cfg)
        missing, unexpected = self.beats.load_state_dict(checkpoint["model"], strict=False)

        if missing:
            print("Missing keys (expected for new predictor):", missing[:5], "..." if len(missing) > 5 else "")
        if unexpected:
            print("Unexpected keys:", unexpected[:5], "..." if len(unexpected) > 5 else "")

    def _encode_frames(self, waveforms: torch.Tensor, padding_mask: Optional[torch.Tensor] = None):
        beats = self.beats
        model_device = next(beats.parameters()).device

        # Важно: считаем fbank на CPU, чтобы обойти CUDA/NVRTC проблемы в torchaudio.kaldi
        waveforms_cpu = waveforms.to("cpu", non_blocking=False)
        fbank = beats.preprocess(waveforms_cpu)

        if padding_mask is not None:
            padding_mask_cpu = padding_mask.to("cpu", non_blocking=False)
            padding_mask = beats.forward_padding_mask(fbank, padding_mask_cpu)

        fbank = fbank.to(model_device, non_blocking=True)
        if padding_mask is not None:
            padding_mask = padding_mask.to(model_device, non_blocking=True)

        fbank = fbank.unsqueeze(1)
        features = beats.patch_embedding(fbank)
        features = features.reshape(features.shape[0], features.shape[1], -1)
        features = features.transpose(1, 2)
        features = beats.layer_norm(features)

        if padding_mask is not None:
            padding_mask = beats.forward_padding_mask(features, padding_mask)

        if beats.post_extract_proj is not None:
            features = beats.post_extract_proj(features)

        x = beats.dropout_input(features)
        x, _ = beats.encoder(x, padding_mask=padding_mask)
        return x, padding_mask

    @staticmethod
    def _masked_mean_pool(frame_embeddings: torch.Tensor, padding_mask: Optional[torch.Tensor]) -> torch.Tensor:
        if padding_mask is not None and padding_mask.any():
            masked = frame_embeddings.masked_fill(padding_mask.unsqueeze(-1), 0.0)
            valid_counts = (~padding_mask).sum(dim=1).clamp(min=1).unsqueeze(-1)
            return masked.sum(dim=1) / valid_counts
        return frame_embeddings.mean(dim=1)

    def extract_embeddings(self, waveforms: torch.Tensor, padding_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        frame_embeddings, padding_mask = self._encode_frames(waveforms, padding_mask=padding_mask)
        return self._masked_mean_pool(frame_embeddings, padding_mask)

    def forward(self, waveforms: torch.Tensor, padding_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        frame_embeddings, padding_mask = self._encode_frames(waveforms, padding_mask=padding_mask)
        x = self.beats.predictor_dropout(frame_embeddings)
        logits_frames = self.beats.predictor(x)

        if padding_mask is not None and padding_mask.any():
            logits_frames = logits_frames.masked_fill(padding_mask.unsqueeze(-1), 0.0)
            valid_counts = (~padding_mask).sum(dim=1).clamp(min=1).unsqueeze(-1)
            logits = logits_frames.sum(dim=1) / valid_counts
        else:
            logits = logits_frames.mean(dim=1)

        return logits


def count_parameters(model: nn.Module) -> Tuple[int, int]:
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


In [24]:
class LoRALinear(nn.Module):
    def __init__(self, base_layer: nn.Linear, r: int = 8, alpha: int = 32, dropout: float = 0.1):
        super().__init__()
        if not isinstance(base_layer, nn.Linear):
            raise TypeError("LoRALinear can wrap only nn.Linear")
        if r <= 0:
            raise ValueError("LoRA rank r must be > 0")

        self.base = base_layer
        self.r = r
        self.scaling = alpha / r
        self.dropout = nn.Dropout(dropout)

        in_features = base_layer.in_features
        out_features = base_layer.out_features
        base_device = base_layer.weight.device
        base_dtype = base_layer.weight.dtype

        self.lora_A = nn.Parameter(
            torch.empty(r, in_features, device=base_device, dtype=base_dtype)
        )
        self.lora_B = nn.Parameter(
            torch.zeros(out_features, r, device=base_device, dtype=base_dtype)
        )

        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)

        # Базовый линейный слой не обучается в LoRA-режиме
        for p in self.base.parameters():
            p.requires_grad = False

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        base_out = self.base(x)
        lora_out = F.linear(F.linear(self.dropout(x), self.lora_A), self.lora_B) * self.scaling
        return base_out + lora_out


def attach_lora_modules(
    root_module: nn.Module,
    target_keywords: Tuple[str, ...],
    r: int,
    alpha: int,
    dropout: float,
) -> List[str]:
    replaced = []

    def _wrap(module: nn.Module, prefix: str = ""):
        for child_name, child in module.named_children():
            full_name = f"{prefix}.{child_name}" if prefix else child_name
            if isinstance(child, nn.Linear) and any(k in full_name for k in target_keywords):
                wrapped = LoRALinear(child, r=r, alpha=alpha, dropout=dropout).to(child.weight.device, dtype=child.weight.dtype)
                setattr(module, child_name, wrapped)
                replaced.append(full_name)
            else:
                _wrap(child, full_name)

    _wrap(root_module)
    return replaced


def configure_training_mode(model: BEATsForSingleLabelClassification, mode: str) -> Dict:
    if mode not in {"full_tuning", "head_tuning", "lora_tuning"}:
        raise ValueError(f"Unknown mode: {mode}")

    # Сначала всё замораживаем
    for p in model.parameters():
        p.requires_grad = False

    lora_replaced = []

    if mode == "full_tuning":
        for p in model.parameters():
            p.requires_grad = True

    elif mode == "head_tuning":
        for p in model.beats.predictor.parameters():
            p.requires_grad = True

    elif mode == "lora_tuning":
        lora_replaced = attach_lora_modules(
            model.beats,
            target_keywords=LORA_TARGET_KEYWORDS,
            r=LORA_R,
            alpha=LORA_ALPHA,
            dropout=LORA_DROPOUT,
        )

        # Обучаем LoRA + классификатор
        for name, p in model.named_parameters():
            if ("lora_A" in name) or ("lora_B" in name) or name.startswith("beats.predictor"):
                p.requires_grad = True

    total, trainable = count_parameters(model)
    print(f"mode={mode} | trainable={trainable:,} / total={total:,} ({100.0 * trainable / total:.2f}%)")
    if lora_replaced:
        print(f"LoRA modules attached: {len(lora_replaced)}")

    return {
        "mode": mode,
        "trainable_params": trainable,
        "total_params": total,
        "lora_modules": lora_replaced,
    }


In [25]:
@dataclass
class EpochStats:
    loss: float
    acc: float
    f1_macro: float


def autocast_context(use_amp: bool, device: str):
    if use_amp and str(device).startswith("cuda"):
        return torch.amp.autocast(device_type="cuda", enabled=True)
    return nullcontext()


def make_grad_scaler(use_amp: bool, device: str):
    enabled = use_amp and str(device).startswith("cuda")
    if hasattr(torch, "amp") and hasattr(torch.amp, "GradScaler"):
        return torch.amp.GradScaler("cuda", enabled=enabled)
    return torch.cuda.amp.GradScaler(enabled=enabled)


def run_epoch(
    model: nn.Module,
    loader: DataLoader,
    device: str,
    criterion: nn.Module,
    optimizer: Optional[torch.optim.Optimizer] = None,
    scaler: Optional[torch.amp.GradScaler] = None,
    use_amp: bool = False,
    grad_clip_norm: Optional[float] = None,
) -> EpochStats:
    is_train = optimizer is not None
    model.train(is_train)

    all_losses = []
    y_true, y_pred = [], []

    pbar = tqdm(loader, desc="train" if is_train else "eval", leave=False)
    for batch in pbar:
        waveforms = batch["waveforms"].to(device, non_blocking=True)
        padding_mask = batch["padding_mask"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        with autocast_context(use_amp=use_amp, device=device):
            logits = model(waveforms, padding_mask=padding_mask)
            loss = criterion(logits, labels)

        if is_train:
            if scaler is not None and use_amp:
                scaler.scale(loss).backward()
                if grad_clip_norm is not None:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                if grad_clip_norm is not None:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)
                optimizer.step()

        preds = torch.argmax(logits, dim=-1)
        y_true.extend(labels.detach().cpu().tolist())
        y_pred.extend(preds.detach().cpu().tolist())
        all_losses.append(float(loss.detach().item()))

    loss_value = float(np.mean(all_losses)) if all_losses else 0.0
    acc = accuracy_score(y_true, y_pred) if y_true else 0.0
    f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0) if y_true else 0.0

    return EpochStats(loss=loss_value, acc=acc, f1_macro=f1_macro)


In [26]:
model = BEATsForSingleLabelClassification(
    checkpoint_path=CHECKPOINT_PATH,
    num_classes=len(label2id),
).to(DEVICE)

/home/evgeniy/miniconda3/envs/vocal_assist_py310/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Missing keys (expected for new predictor): ['predictor.weight', 'predictor.bias'] 


In [27]:
from torchinfo import summary

summary(model, col_names=["num_params", "trainable"])

Layer (type:depth-idx)                                       Param #                   Trainable
BEATsForSingleLabelClassification                            --                        True
├─BEATs: 1-1                                                 --                        True
│    └─Linear: 2-1                                           393,984                   True
│    └─Conv2d: 2-2                                           131,072                   True
│    └─Dropout: 2-3                                          --                        --
│    └─TransformerEncoder: 2-4                               --                        True
│    │    └─Sequential: 3-1                                  4,719,488                 True
│    │    └─ModuleList: 3-2                                  85,064,688                True
│    │    └─LayerNorm: 3-3                                   1,536                     True
│    └─LayerNorm: 2-5                                        1,024           

In [28]:
def init_comet_experiment(run_name: str, output_dir: Path, mode: str):
    if not COMET_ENABLED:
        return None

    try:
        from comet_ml import Experiment, OfflineExperiment
    except Exception as e:
        print(f"Comet disabled: comet_ml is not available ({e})")
        return None

    api_key = os.getenv("COMET_API_KEY", "").strip()
    workspace = COMET_WORKSPACE

    try:
        if api_key:
            exp = Experiment(
                api_key=api_key,
                project_name=COMET_PROJECT_NAME,
                workspace=workspace,
                auto_output_logging="simple",
                log_code=False,
            )
            print("Comet: online experiment started")
        else:
            if not COMET_AUTO_OFFLINE:
                print("Comet disabled: COMET_API_KEY not set and COMET_AUTO_OFFLINE=False")
                return None
            offline_dir = output_dir / "comet_offline"
            offline_dir.mkdir(parents=True, exist_ok=True)
            exp = OfflineExperiment(
                project_name=COMET_PROJECT_NAME,
                workspace=workspace,
                offline_directory=str(offline_dir),
                auto_output_logging="simple",
                log_code=False,
            )
            print(f"Comet: offline experiment started at {offline_dir}")

        exp.set_name(f"{COMET_EXPERIMENT_PREFIX}_{run_name}")
        return exp
    except Exception as e:
        print(f"Comet disabled: failed to init experiment ({e})")
        return None


def log_comet_metrics(exp, metrics: Dict[str, float], step: Optional[int] = None, epoch: Optional[int] = None):
    if exp is None:
        return
    clean = {}
    for k, v in metrics.items():
        if isinstance(v, (int, float, np.floating, np.integer)):
            clean[k] = float(v)
    if clean:
        exp.log_metrics(clean, step=step, epoch=epoch)


In [29]:
def train_mode(mode: str):
    run_name = f"beats_annotated_{mode}"
    output_dir = OUTPUT_BASE_DIR / run_name
    output_dir.mkdir(parents=True, exist_ok=True)

    model = BEATsForSingleLabelClassification(
        checkpoint_path=CHECKPOINT_PATH,
        num_classes=len(label2id),
    )

    mode_info = configure_training_mode(model, mode)
    model = model.to(DEVICE)

    trainable_params = [p for p in model.parameters() if p.requires_grad]
    if not trainable_params:
        raise RuntimeError("Нет trainable параметров. Проверьте режим обучения.")

    optimizer = torch.optim.AdamW(
        trainable_params,
        lr=LR_BY_MODE[mode],
        weight_decay=WEIGHT_DECAY,
    )
    criterion = nn.CrossEntropyLoss()

    epochs = EPOCHS_BY_MODE[mode]
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, epochs))

    use_amp = USE_AMP and (DEVICE == "cuda")
    scaler = make_grad_scaler(use_amp=use_amp, device=DEVICE)

    experiment = init_comet_experiment(run_name=run_name, output_dir=output_dir, mode=mode)
    if experiment is not None:
        experiment.log_parameters({
            "mode": mode,
            "seed": SEED,
            "device": DEVICE,
            "batch_size": BATCH_SIZE,
            "epochs": epochs,
            "lr": LR_BY_MODE[mode],
            "weight_decay": WEIGHT_DECAY,
            "eval_every_steps": EVAL_EVERY_STEPS,
            "max_audio_seconds": MAX_AUDIO_SECONDS,
            "target_sr": TARGET_SR,
            "lora_r": LORA_R,
            "lora_alpha": LORA_ALPHA,
            "lora_dropout": LORA_DROPOUT,
            "train_size": len(train_df),
            "val_size": len(val_df),
            "num_classes": len(label2id),
            "trainable_params": mode_info["trainable_params"],
            "total_params": mode_info["total_params"],
        })

    best_f1 = -1.0
    best_state = None
    best_epoch = None
    best_step = None
    best_ckpt_path = output_dir / "best_model.pt"

    history = []
    step_eval_history = []
    global_step = 0

    for epoch in range(1, epochs + 1):
        model.train(True)
        train_losses = []
        recent_train_losses = []
        y_true_train, y_pred_train = [], []

        pbar = tqdm(train_loader, desc=f"train epoch: {epoch}", leave=False)
        for batch in pbar:
            waveforms = batch["waveforms"].to(DEVICE, non_blocking=True)
            padding_mask = batch["padding_mask"].to(DEVICE, non_blocking=True)
            labels = batch["labels"].to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with autocast_context(use_amp=use_amp, device=DEVICE):
                logits = model(waveforms, padding_mask=padding_mask)
                loss = criterion(logits, labels)

            if scaler is not None and use_amp:
                scaler.scale(loss).backward()
                if GRAD_CLIP_NORM is not None:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                if GRAD_CLIP_NORM is not None:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                optimizer.step()

            preds = torch.argmax(logits, dim=-1)
            loss_item = float(loss.detach().item())
            train_losses.append(loss_item)
            recent_train_losses.append(loss_item)
            if EVAL_EVERY_STEPS and len(recent_train_losses) > EVAL_EVERY_STEPS:
                recent_train_losses.pop(0)

            y_true_train.extend(labels.detach().cpu().tolist())
            y_pred_train.extend(preds.detach().cpu().tolist())
            global_step += 1

            if experiment is not None and (global_step % COMET_LOG_EVERY_N_STEPS == 0):
                log_comet_metrics(
                    experiment,
                    {
                        "train/step_loss": loss_item,
                        "train/lr": float(optimizer.param_groups[0]["lr"]),
                    },
                    step=global_step,
                    epoch=epoch,
                )

            if EVAL_EVERY_STEPS and (global_step % EVAL_EVERY_STEPS == 0):
                val_stats_step = run_epoch(
                    model,
                    val_loader,
                    device=DEVICE,
                    criterion=criterion,
                    optimizer=None,
                    scaler=None,
                    use_amp=use_amp,
                )

                train_loss_window = float(np.mean(recent_train_losses)) if recent_train_losses else float('nan')
                step_row = {
                    "epoch": epoch,
                    "global_step": global_step,
                    "train_loss_window": train_loss_window,
                    "val_loss": val_stats_step.loss,
                    "val_acc": val_stats_step.acc,
                    "val_f1_macro": val_stats_step.f1_macro,
                }
                step_eval_history.append(step_row)

                print(
                    f"[{mode}] step {global_step} | "
                    f"train_loss(avg{EVAL_EVERY_STEPS})={train_loss_window:.4f} | "
                    f"val_loss={val_stats_step.loss:.4f} val_acc={val_stats_step.acc:.4f} val_f1={val_stats_step.f1_macro:.4f}"
                )

                log_comet_metrics(
                    experiment,
                    {
                        "train/step_loss_window": train_loss_window,
                        "val/step_loss": val_stats_step.loss,
                        "val/step_acc": val_stats_step.acc,
                        "val/step_f1_macro": val_stats_step.f1_macro,
                    },
                    step=global_step,
                    epoch=epoch,
                )

                if val_stats_step.f1_macro > best_f1:
                    best_f1 = val_stats_step.f1_macro
                    best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
                    best_epoch = epoch
                    best_step = global_step

                model.train(True)

        train_loss = float(np.mean(train_losses)) if train_losses else 0.0
        train_acc = accuracy_score(y_true_train, y_pred_train) if y_true_train else 0.0
        train_f1 = f1_score(y_true_train, y_pred_train, average="macro", zero_division=0) if y_true_train else 0.0
        train_stats = EpochStats(loss=train_loss, acc=train_acc, f1_macro=train_f1)

        val_stats = run_epoch(
            model,
            val_loader,
            device=DEVICE,
            criterion=criterion,
            optimizer=None,
            scaler=None,
            use_amp=use_amp,
        )

        scheduler.step()

        epoch_row = {
            "epoch": epoch,
            "global_step": global_step,
            "train_loss": train_stats.loss,
            "train_acc": train_stats.acc,
            "train_f1_macro": train_stats.f1_macro,
            "val_loss": val_stats.loss,
            "val_acc": val_stats.acc,
            "val_f1_macro": val_stats.f1_macro,
            "lr": optimizer.param_groups[0]["lr"],
        }
        history.append(epoch_row)

        print(
            f"[{mode}] Epoch {epoch:02d}/{epochs} | "
            f"train_loss={train_stats.loss:.4f} train_f1={train_stats.f1_macro:.4f} | "
            f"val_loss={val_stats.loss:.4f} val_acc={val_stats.acc:.4f} val_f1={val_stats.f1_macro:.4f}"
        )

        log_comet_metrics(
            experiment,
            {
                "train/epoch_loss": train_stats.loss,
                "train/epoch_acc": train_stats.acc,
                "train/epoch_f1_macro": train_stats.f1_macro,
                "val/epoch_loss": val_stats.loss,
                "val/epoch_acc": val_stats.acc,
                "val/epoch_f1_macro": val_stats.f1_macro,
                "train/lr_epoch": optimizer.param_groups[0]["lr"],
            },
            step=global_step,
            epoch=epoch,
        )

        if val_stats.f1_macro > best_f1:
            best_f1 = val_stats.f1_macro
            best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
            best_epoch = epoch
            best_step = global_step

    if best_state is not None:
        torch.save(
            {
                "state_dict": best_state,
                "label2id": label2id,
                "id2label": id2label,
                "mode": mode,
                "metadata_csv": str(metadata_path),
                "checkpoint_path": str(CHECKPOINT_PATH),
                "mode_info": mode_info,
                "config": {
                    "target_sr": TARGET_SR,
                    "max_audio_seconds": MAX_AUDIO_SECONDS,
                    "batch_size": BATCH_SIZE,
                    "epochs": epochs,
                    "lr": LR_BY_MODE[mode],
                    "weight_decay": WEIGHT_DECAY,
                    "lora_r": LORA_R,
                    "lora_alpha": LORA_ALPHA,
                    "lora_dropout": LORA_DROPOUT,
                    "eval_every_steps": EVAL_EVERY_STEPS,
                },
                "best_epoch": best_epoch,
                "best_step": best_step,
            },
            best_ckpt_path,
        )
        model.load_state_dict(best_state, strict=True)
        print("Saved and loaded best checkpoint:", best_ckpt_path)

    history_df = pd.DataFrame(history)
    history_df.to_csv(output_dir / "history.csv", index=False)

    step_eval_df = pd.DataFrame(step_eval_history)
    step_eval_df.to_csv(output_dir / "step_eval_history.csv", index=False)

    final_metrics = {
        "mode": mode,
        "best_val_f1_macro": best_f1,
        "best_epoch": best_epoch,
        "best_step": best_step,
        "trainable_params": mode_info["trainable_params"],
        "total_params": mode_info["total_params"],
        "metadata_csv": str(metadata_path),
        "checkpoint_path": str(CHECKPOINT_PATH),
        "best_checkpoint_path": str(best_ckpt_path),
        "eval_every_steps": EVAL_EVERY_STEPS,
    }

    with open(output_dir / "metrics.json", "w", encoding="utf-8") as f:
        json.dump(final_metrics, f, ensure_ascii=False, indent=2)

    if experiment is not None:
        log_comet_metrics(
            experiment,
            {
                "best/val_f1_macro": best_f1,
                "best/epoch": float(best_epoch) if best_epoch is not None else -1.0,
                "best/step": float(best_step) if best_step is not None else -1.0,
            },
            step=global_step,
            epoch=best_epoch,
        )
        experiment.log_asset(str(output_dir / "metrics.json"))
        experiment.log_asset(str(output_dir / "history.csv"))
        experiment.log_asset(str(output_dir / "step_eval_history.csv"))
        experiment.end()

    print("Saved metrics/history to:", output_dir)
    return {
        "model": model,
        "history": history_df,
        "step_eval_history": step_eval_df,
        "metrics": final_metrics,
        "output_dir": output_dir,
    }


In [ ]:
modes_to_run = MODES if RUN_ALL_MODES else [TRAINING_MODE]
results = {}

for mode in modes_to_run:
    print("\n" + "=" * 80)
    print(f"Start mode: {mode}")
    print("=" * 80)
    results[mode] = train_mode(mode)

print("\nCompleted modes:", list(results.keys()))



Start mode: head_tuning


COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: sklearn, torch.


Missing keys (expected for new predictor): ['predictor.weight', 'predictor.bias'] 
mode=head_tuning | trainable=12,304 / total=90,324,096 (0.01%)


COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : beats_beats_annotated_lora_tuning
COMET INFO:     url                   : https://www.comet.com/13evgeniymarchuk89/diploma/62ed0ca643e84fbc807579367cd15443
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     train/epoch_acc [3]         : (0.3403185247275775, 0.6424979044425817)
COMET INFO:     train/epoch_f1_macro [3]    : (0.1758370875727192, 0.5630549832462701)
COMET INFO:     train/epoch_loss [3]        : (0.9763952420909001, 1.9781661234309327)
COMET INFO:     train/lr [400]              : (7.564026096753472e-05, 8e-05)
COMET INFO:     train/lr_epoch [3]          : (7.564026096753472e-05, 7.950753362380551e-05)
COMET INFO:     train/step_lo

Comet: online experiment started


...

eval:   0%|          | 0/70 [00:00<?, ?it/s]

[head_tuning] Epoch 03/20 | train_loss=1.8065 train_f1=0.2678 | val_loss=1.6693 val_acc=0.5036 val_f1=0.2976


train epoch: 4:   0%|          | 0/597 [00:00<?, ?it/s]

eval:   0%|          | 0/70 [00:00<?, ?it/s]

[head_tuning] step 1840 | train_loss(avg80)=1.7306 | val_loss=1.6549 val_acc=0.5216 val_f1=0.3131


In [ ]:
def evaluate_full_validation_set(
    model: BEATsForSingleLabelClassification,
    loader: DataLoader,
    device: str,
    criterion: nn.Module,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    model.eval()
    losses = []
    y_true, y_pred = [], []

    use_amp = USE_AMP and str(device).startswith("cuda")

    with torch.no_grad():
        for batch in tqdm(loader, desc="final val eval", leave=False):
            waveforms = batch["waveforms"].to(device, non_blocking=True)
            padding_mask = batch["padding_mask"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True)

            with autocast_context(use_amp=use_amp, device=device):
                logits = model(waveforms, padding_mask=padding_mask)
                loss = criterion(logits, labels)

            preds = torch.argmax(logits, dim=-1)
            losses.append(float(loss.detach().item()))
            y_true.extend(labels.detach().cpu().tolist())
            y_pred.extend(preds.detach().cpu().tolist())

    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)

    metrics = {
        "val_loss": float(np.mean(losses)) if losses else 0.0,
        "val_acc": float(accuracy_score(y_true, y_pred)) if len(y_true) else 0.0,
        "val_f1_macro": float(f1_score(y_true, y_pred, average="macro", zero_division=0)) if len(y_true) else 0.0,
    }
    return metrics, y_true, y_pred


def extract_embeddings_for_tsne(
    model: BEATsForSingleLabelClassification,
    loader: DataLoader,
    device: str,
) -> Tuple[np.ndarray, np.ndarray]:
    model.eval()
    all_embeddings = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(loader, desc="extract embeddings", leave=False):
            waveforms = batch["waveforms"].to(device, non_blocking=True)
            padding_mask = batch["padding_mask"].to(device, non_blocking=True)
            labels = batch["labels"].cpu().numpy().astype(np.int64)

            embeddings = model.extract_embeddings(waveforms, padding_mask=padding_mask)
            all_embeddings.append(embeddings.detach().float().cpu().numpy())
            all_labels.append(labels)

    return np.concatenate(all_embeddings, axis=0), np.concatenate(all_labels, axis=0)


def plot_tsne_embeddings(
    embeddings: np.ndarray,
    labels: np.ndarray,
    id2label_map: Dict[int, str],
    title: str,
    save_path: Path,
    max_points: int = 2500,
    random_state: int = 42,
):
    n = embeddings.shape[0]
    idx = np.arange(n)
    if n > max_points:
        rng = np.random.default_rng(random_state)
        idx = rng.choice(n, size=max_points, replace=False)

    emb = embeddings[idx]
    y = labels[idx]

    perplexity = min(30, max(5, (len(emb) - 1) // 3))
    tsne = TSNE(
        n_components=2,
        perplexity=perplexity,
        learning_rate="auto",
        init="pca",
        random_state=random_state,
    )
    emb2d = tsne.fit_transform(emb)

    classes = sorted(np.unique(y).tolist())
    cmap = plt.cm.get_cmap("tab20", max(len(classes), 1))

    plt.figure(figsize=(12, 9))
    for i, cls in enumerate(classes):
        mask = y == cls
        label_name = id2label_map.get(int(cls), f"class_{cls}")
        plt.scatter(
            emb2d[mask, 0],
            emb2d[mask, 1],
            s=18,
            alpha=0.75,
            color=cmap(i),
            label=label_name,
        )

    plt.title(title)
    plt.xlabel("t-SNE dim 1")
    plt.ylabel("t-SNE dim 2")
    plt.legend(loc="best", fontsize=8, ncol=2)
    plt.tight_layout()

    save_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(save_path, dpi=220)
    plt.show()


# Финальная валидация + t-SNE после обучения
if results:
    mode0 = list(results.keys())[0]
    trained_model = results[mode0]["model"]
    output_dir = results[mode0]["output_dir"]

    criterion = nn.CrossEntropyLoss()
    val_metrics, y_true, y_pred = evaluate_full_validation_set(
        trained_model,
        val_loader,
        device=DEVICE,
        criterion=criterion,
    )

    print("Final validation metrics:", val_metrics)

    report = classification_report(
        y_true,
        y_pred,
        labels=sorted(id2label.keys()),
        target_names=[id2label[i] for i in sorted(id2label.keys())],
        zero_division=0,
        output_dict=True,
    )
    report_df = pd.DataFrame(report).T
    report_df.to_csv(output_dir / "val_classification_report_final.csv")

    cm = confusion_matrix(y_true, y_pred, labels=sorted(id2label.keys()))
    cm_df = pd.DataFrame(
        cm,
        index=[id2label[i] for i in sorted(id2label.keys())],
        columns=[id2label[i] for i in sorted(id2label.keys())],
    )
    cm_df.to_csv(output_dir / "val_confusion_matrix_final.csv")

    metrics_path = output_dir / "val_metrics_final.json"
    with open(metrics_path, "w", encoding="utf-8") as f:
        json.dump(val_metrics, f, ensure_ascii=False, indent=2)

    display(pd.DataFrame([val_metrics]))
    display(report_df.head(12))

    val_embeddings, val_labels = extract_embeddings_for_tsne(trained_model, val_loader, device=DEVICE)

    tsne_dir = PROJECT_ROOT / "MainExperiments" / "tsne_plots"
    tsne_path = tsne_dir / f"tsne_beats_annotated_{mode0}.png"

    plot_tsne_embeddings(
        embeddings=val_embeddings,
        labels=val_labels,
        id2label_map=id2label,
        title=f"BEATs t-SNE ({mode0})",
        save_path=tsne_path,
        max_points=2500,
        random_state=SEED,
    )
    print(f"Saved t-SNE plot to: {tsne_path}")


def predict_single_file(
    model: BEATsForSingleLabelClassification,
    audio_path: str,
    id2label_map: Dict[int, str],
    target_sr: int = 16000,
    max_audio_seconds: Optional[float] = 8.0,
) -> Tuple[str, float]:
    wav, sr = load_audio_robust(audio_path)
    if wav.size(0) > 1:
        wav = wav.mean(dim=0, keepdim=True)
    wav = wav.squeeze(0)

    if sr != target_sr:
        wav = torchaudio.transforms.Resample(sr, target_sr)(wav.unsqueeze(0)).squeeze(0)

    peak = wav.abs().max().clamp(min=1e-6)
    wav = wav / peak

    if max_audio_seconds is not None:
        max_samples = int(max_audio_seconds * target_sr)
        if wav.numel() > max_samples:
            start = (wav.numel() - max_samples) // 2
            wav = wav[start : start + max_samples]

    waveforms = wav.unsqueeze(0).to(DEVICE)
    padding_mask = torch.zeros_like(waveforms, dtype=torch.bool, device=DEVICE)

    model.eval()
    with torch.no_grad():
        logits = model(waveforms, padding_mask=padding_mask)
        probs = torch.softmax(logits, dim=-1)
        conf, pred_id = torch.max(probs, dim=-1)

    return id2label_map[int(pred_id.item())], float(conf.item())


# Пример предсказаний на нескольких валидационных записях
if results:
    mode0 = list(results.keys())[0]
    trained_model = results[mode0]["model"]

    sample_df = val_df.sample(n=min(5, len(val_df)), random_state=SEED)
    rows = []
    for _, row in sample_df.iterrows():
        pred_label, conf = predict_single_file(
            trained_model,
            row["path"],
            id2label,
            target_sr=TARGET_SR,
            max_audio_seconds=MAX_AUDIO_SECONDS,
        )
        rows.append(
            {
                "path": row["path"],
                "true_label": row[LABEL_COL],
                "pred_label": pred_label,
                "confidence": conf,
            }
        )

    pred_df = pd.DataFrame(rows)
    display(pred_df)


## Notes

- Если хотите прогнать все режимы за один запуск: `RUN_ALL_MODES = True`.
- Если не хватает памяти GPU, уменьшите `BATCH_SIZE` и/или `MAX_AUDIO_SECONDS`.
- Для более быстрой проверки можно временно уменьшить `EPOCHS_BY_MODE`.
